In [2]:
pip install searoute

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 15.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for searoute: filename=searoute-1.4.2-py3-none-any.whl size=1033863 sha256=43dc95f7ec950ff81ba4379c66a1b1ea277ce08b25e57803b6c6425f2771a320
  Stored in directory: /Users/tomasss/Library/Caches/pip/wheels/a9/78/ca/78eefc783e5748d57ad435dee2f9c8e0c3c760cfb9c7e88a61
Successfully built searoute

[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import searoute as sr

def calculate_maritime_route(origin, destination, units="km", speed_knot=24, 
                             append_orig_dest=True, restrictions=None, 
                             include_ports=True, port_params=None, 
                             return_passages=True):
    """
    Calculates a maritime route and returns routing info and map in GeoJSON format.
    
    Args:
        origin (tuple): Coordinates of the origin point (longitude, latitude).
        destination (tuple): Coordinates of the destination point (longitude, latitude).
        units (str): Units for distance ('km', 'm', 'mi', 'naut', etc.). Defaults to 'km'.
        speed_knot (float): Speed in knots. Defaults to 20.
        append_orig_dest (bool): Whether to include origin/destination in the route. Defaults to True.
        restrictions (list): List of restricted passages (e.g., ['northwest']). Defaults to None.
        include_ports (bool): Whether to include nearby ports in the calculation. Defaults to True.
        port_params (dict): Parameters for port selection (e.g., {'only_terminals': True}). Defaults to None.
        return_passages (bool): Whether to return passage information. Defaults to True.
    
    Returns:
        dict: Route information with GeoJSON map.
    """
    import logging

    logging.basicConfig(level=logging.DEBUG)
    logger = logging.getLogger(__name__)

    logger.debug(f"Inputs - Origin: {origin}, Destination: {destination}, Units: {units}, Speed: {speed_knot}")
    logger.debug(f"Restrictions: {restrictions}, Include Ports: {include_ports}, Port Params: {port_params}")

    if restrictions is None:
        restrictions = ['northwest']
    
    if port_params is None:
        port_params = {'only_terminals': False}
    
    # Calculate the route
    try:
        route = sr.searoute(origin, 
                            destination, 
                            units=units, 
                            speed_knot=speed_knot, 
                            append_orig_dest=append_orig_dest, 
                            restrictions=restrictions, 
                            include_ports=include_ports, 
                            port_params=port_params, 
                            return_passages=return_passages)
        
        # Retrieve distance and units
        distance = route.properties.get('length', 0)
        distance_units = route.properties.get('units', units)
        
        # Print route information
        print(f"Route length: {distance:.1f} {distance_units}")
        
        return {
            "route_geojson": route,  # GeoJSON LineString Feature
            "distance": distance,
            "units": distance_units
        }
    except Exception as e:
        print(f"Error calculating route: {e}")
        return None

# Example Usage
origin = (-121.25, -58.74)  # Example origin
destination = (-123.75, -58.74)  # Example destination
result = calculate_maritime_route(
    origin, 
    destination, 
    units="naut", 
    speed_knot=15, 
    restrictions=['northwest'], 
    port_params={'only_terminals': True, 'country_pol': 'FR', 'country_pod': 'CN'}
)

# Access and use the results
if result:
    print(f"Distance: {result['distance']} {result['units']}")
    print(f"GeoJSON Map: {result['route_geojson']}")


DEBUG:__main__:Inputs - Origin: (-121.25, -58.74), Destination: (-123.75, -58.74), Units: naut, Speed: 15
DEBUG:__main__:Restrictions: ['northwest'], Include Ports: True, Port Params: {'only_terminals': True, 'country_pol': 'FR', 'country_pod': 'CN'}


Route length: 32290.0 naut
Distance: 32290.043981697323 naut
GeoJSON Map: {"geometry": {"coordinates": [[-121.25, -58.74], [-4.490662, 48.386768], [-120, -50], [-251.364106, 19.087023], [-123.75, -58.74]], "type": "LineString"}, "properties": {"duration_hours": 2152.6695987798216, "length": 32290.043981697323, "port_dest": {"cty": "China", "name": "Basou", "port": "CNBAS", "t": 1.0, "x": 108.635894, "y": 19.087023}, "port_origin": {"cty": "France", "name": "Brest", "port": "FRBES", "t": 1.0, "x": -4.490662, "y": 48.386768}, "traversed_passages": [], "units": "naut"}, "type": "Feature"}


In [9]:
calculate_maritime_route((-121.25, -58.74),(-123.75, -58.74))

DEBUG:__main__:Inputs - Origin: (-121.25, -58.74), Destination: (-123.75, -58.74), Units: km, Speed: 24
DEBUG:__main__:Restrictions: None, Include Ports: True, Port Params: None


Route length: 12701.6 km


{'route_geojson': {"geometry": {"coordinates": [[-121.25, -58.74], [-109.43558, -27.155651], [-120, -50], [-109.43558, -27.155651], [-123.75, -58.74]], "type": "LineString"}, "properties": {"duration_hours": 285.76352380732936, "length": 12701.617106188176, "port_dest": {"cty": "Chile", "name": "Isla de Pascua", "port": "CLIPC", "t": null, "x": -109.43558, "y": -27.155651}, "port_origin": {"cty": "Chile", "name": "Isla de Pascua", "port": "CLIPC", "t": null, "x": -109.43558, "y": -27.155651}, "traversed_passages": [], "units": "km"}, "type": "Feature"},
 'distance': 12701.617106188176,
 'units': 'km'}